# Acidentes em rodovias federais (PRF, 2007–2020) — Análise exploratória
**Disciplina:** Linguagem de Programação (Análise e Visualização de Dados com Python) · 6º período · BSI
**Autores:** Lucas de Almeida Brugger Cavalcante, *[integrante 2]*, *[integrante 3]* · **Instituição:** *[preencher]*

Este notebook reproduz a análise do artigo (`docs/index.html`). Ele lê os dados **já tratados** por `src/etl.py`
(`data/processed/acidentes_prf_tratado.csv.gz`) e responde às três perguntas de pesquisa:

* **RQ1** – Como evoluíram acidentes, mortos e letalidade entre 2007 e 2020?
* **RQ2** – Que causas, tipos e condições se associam à maior gravidade?
* **RQ3** – Onde (UF e trechos) as mortes se concentram?

In [ ]:
# --- Preparação do ambiente (funciona localmente e no Google Colab) ---
import os, sys, subprocess

REPO_URL = "https://github.com/Almeidz/projeto-acidentes-prf.git"   # ajuste se o nome do repositório for outro

if "google.colab" in sys.modules and not os.path.exists("src"):
    subprocess.run(["git", "clone", REPO_URL, "projeto"], check=True)
    os.chdir("projeto")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                                                   # raiz do repositório
sys.path.insert(0, os.getcwd())

import pandas as pd
import matplotlib.pyplot as plt
from src import analise as a, figuras as f

pd.set_option("display.max_columns", 30, "display.width", 160)

## 1. Carga e auditoria dos dados tratados

In [ ]:
df = a.carregar()
print(f"{len(df):,} acidentes · {df['ano'].min()}–{df['ano'].max()} · anos presentes: {sorted(int(x) for x in df['ano'].unique())}")
df.head()

In [ ]:
# Rastreabilidade do ETL: o que foi lido, descartado e anulado em cada arquivo
import json
log = json.load(open("data/processed/etl_log.json", encoding="utf-8"))
pd.DataFrame({k: v for k, v in log.items() if k != "_total"}).T

In [ ]:
# Valores ausentes e estatística descritiva (com assimetria) das variáveis numéricas
display(df.isna().sum()[lambda s: s > 0].to_frame("ausentes"))
display(a.resumo_numerico(df).round(2))

**Leitura:** `pessoas`, `feridos` e `mortos` têm assimetria alta (poucos acidentes muito grandes — máx. de 248 pessoas e 33 mortos
em uma ocorrência). Esses *outliers* são reais (ônibus, engavetamentos), por isso **foram mantidos**; usamos soma/proporção e
evitamos médias por acidente. `latitude/longitude` só existem a partir de 2017 (≈87% ausentes na série completa).

## 2. RQ1 — Evolução temporal

In [ ]:
anual = a.serie_anual(df)
pct_sem = df.assign(sv=df["gravidade"].eq("sem_vitimas")).groupby("ano")["sv"].mean() * 100
display(anual.assign(pct_sem_vitimas=pct_sem.values).round(2))
f.fig_anual(anual); plt.show()

In [ ]:
# A letalidade sobe, mas a parcela de acidentes sem vítimas despenca: indício de mudança no registro
f.fig_sem_vitimas(anual, pct_sem); plt.show()

## 3. RQ2 — Fatores associados à gravidade

In [ ]:
causas = a.por_causa(df)
display(causas.round(2))
f.fig_causas(causas); plt.show()

In [ ]:
tipos = a.por_tipo(df)
display(tipos.round(2).head(8))
f.fig_tipos(tipos); plt.show()

In [ ]:
h = a.heatmap_dia_hora(df)
f.fig_heatmap(h); plt.show()
dia, hora = h.stack().idxmax()
print(f"Pico: {dia} às {int(hora):02d}h com {int(h.loc[dia, hora]):,} mortes")

In [ ]:
fase, clima = a.por_fase_dia(df), a.por_clima(df)
display(fase.round(2)); display(clima.round(2))
f.fig_fase_clima(fase, clima); plt.show()

In [ ]:
# Teste de independência (qui-quadrado) entre cada fator e ocorrência de morte.
# Com n ~ 1,8 milhão o p-valor é sempre ~0; o que informa é o tamanho de efeito (V de Cramér).
pd.DataFrame({c: a.teste_qui2(df, c) for c in ["fase_dia", "tipo_pista", "condicao_meteorologica", "dia_semana"]}).T

## 4. RQ3 — Distribuição geográfica

In [ ]:
uf = a.por_uf(df)
display(uf.round(2))
f.fig_uf(uf); plt.show()

In [ ]:
from scipy import stats
rho, p = stats.spearmanr(uf["acidentes"], uf["letalidade"])
print(f"Spearman (acidentes × letalidade, por UF): rho = {rho:.2f}, p = {p:.4f}")
trechos = a.trechos_criticos(df)
display(trechos)
f.fig_trechos(trechos); plt.show()

## 5. Conclusões e limitações
* **RQ1:** acidentes e mortes caíram, mas em proporções diferentes; a alta de letalidade coincide com a queda da parcela de
  acidentes sem vítimas → provável mudança de registro (hipótese; os dados não permitem separar de uma queda real).
* **RQ2:** maior gravidade em colisões frontais e atropelamentos, à noite e nos fins de semana (pico: domingo às 18h).
* **RQ3:** MG e BA lideram em mortes absolutas; MA e TO em letalidade; trechos críticos em regiões metropolitanas.
* **Limitações:** 2019 ausente; taxonomia de causas muda em 2017; sem normalização por tráfego/extensão; coordenadas só pós-2017.

O artigo completo (com gráficos interativos Plotly) está em `docs/index.html`.